# Evaluation — Wikidata

**Purpose:** Evaluates retrieval rankings from the trained Hybrid SPARQL encoder and baselines (Qwen, MiniLM, E5, MPNet, TF-IDF). Relevance is graded by result-set Jaccard (no oracle).

**Input:** `../LSQ_queries/semantic_triplets/semantic_triplets_wikidata.json`, `../Train/splits/semantic_triplet_splits_wikidata.json`, `../checkpoints/wikidata_model_semantic-20.pt`.

**Metrics:** Avg Jaccard@k, nDCG@k (graded); P/R/F1@k (binary relevance with Jaccard threshold).

### 1. Imports

In [17]:
import json
import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from typing import Dict, Any, List, Set, Tuple

from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize
from scipy.stats import spearmanr
from src.inference import SPARQLEmbedder


In [2]:
SEMANTIC_PATH = "../LSQ_queries/semantic_triplets/semantic_triplets_wikidata.json" 
SPLITS_PATH = "../Train/splits/semantic_triplet_splits_wikidata.json"
CHECKPOINT_PATH = "../checkpoints/wikidata_model_semantic-20.pt"

QWEN_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MINILM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
E5_MODEL_NAME = "intfloat/e5-base-v2"
MPNET_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

RESULT_LIMIT = 200
TOPKS = [3, 5, 10]
BATCH = 32
JACCARD_BINARY_THRESHOLD = 0.8


In [3]:
with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    raw_sem = json.load(f)

with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    split_info = json.load(f)

test_idx = split_info.get("test_idx", split_info.get("valid_idx"))
print(f"Loaded {len(raw_sem)} semantic entries | Eval size: {len(test_idx)}")


Loaded 10735 semantic entries | Eval size: 1074


### 3. Load data and splits

In [4]:
endpoint = SPARQLWrapper("https://wikidata.data.dice-research.org/sparql")
endpoint.setReturnFormat(SPARQL_JSON)

answer_cache: Dict[str, Set[str]] = {}

def normalize_query(q: str) -> str:
    return "\n".join(line.strip() for line in q.strip().splitlines())

def get_answers(query: str, limit: int = 200) -> Set[str]:
    q_norm = normalize_query(query)
    if q_norm in answer_cache:
        return answer_cache[q_norm]

    endpoint.setQuery(query + f"\nLIMIT {limit}")
    try:
        res = endpoint.query().convert()
        bindings = res.get("results", {}).get("bindings", [])
    except Exception:
        bindings = []
        res = {"head": {"vars": []}}

    if not bindings:
        answers: Set[str] = set()
    else:
        vars_ = res.get("head", {}).get("vars", [])
        answers = set()
        for b in bindings:
            vals = []
            for v in vars_:
                if v in b:
                    vals.append(b[v]["value"])
            if vals:
                answers.add("|".join(vals))

    answer_cache[q_norm] = answers
    return answers

def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0


In [5]:
def get_anchor_query(item: Dict[str, Any]) -> str:
    return item.get("anchor_query") or item.get("base_query") or ""

def build_pool_queries_for_item(item: Dict[str, Any], positives_only: bool = True) -> List[str]:
    pool: List[str] = []
    for p in item.get("positive_queries", []):
        q = p.get("query") if isinstance(p, dict) else p
        if q:
            pool.append(q)
    if not positives_only:
        for n in item.get("negative_queries", []):
            q = n.get("query") if isinstance(n, dict) else n
            if q:
                pool.append(q)
    seen = set()
    out = []
    for q in pool:
        if q not in seen:
            out.append(q)
            seen.add(q)
    return out

eval_items: List[Tuple[str, List[str]]] = []
unique_queries: List[str] = []
seen_q = set()

def add_unique(q: str):
    if q and q not in seen_q:
        seen_q.add(q)
        unique_queries.append(q)

for idx in test_idx:
    item = raw_sem[idx]
    anchor_q = get_anchor_query(item)
    if not anchor_q:
        continue
    pool = build_pool_queries_for_item(item, positives_only=False)
    # Full pool (positives + negatives) so ranking differs by model; otherwise P/R/F1 are identical when pool is small.
    if not pool:
        continue

    eval_items.append((anchor_q, pool))
    add_unique(anchor_q)
    for q in pool:
        add_unique(q)

print(f"Eval items with non-empty pool: {len(eval_items)}")
print(f"Unique queries to execute/encode: {len(unique_queries)}")


Eval items with non-empty pool: 1074
Unique queries to execute/encode: 3869


In [6]:
print("Executing SPARQL for unique queries (cached)...")
for q in unique_queries:
    _ = get_answers(q, limit=RESULT_LIMIT)

answers_map: Dict[str, Set[str]] = {q: answer_cache[normalize_query(q)] for q in unique_queries}
print("Cached answers:", len(answers_map))


Executing SPARQL for unique queries (cached)...
Cached answers: 3869


### 5. Eval items and unique queries

In [7]:
jaccard_scores_per_item: List[Dict[str, float]] = []

for anchor_q, pool in eval_items:
    a_res = answers_map.get(anchor_q, set())
    rel = {}
    for q in pool:
        rel[q] = jaccard(a_res, answers_map.get(q, set()))
    jaccard_scores_per_item.append(rel)

pos_sets: List[Set[str]] = []
kept_mask: List[bool] = []
for (anchor_q, pool), rel_map in zip(eval_items, jaccard_scores_per_item):
    pos = {q for q in pool if rel_map[q] >= JACCARD_BINARY_THRESHOLD}
    pos_sets.append(pos)
    kept_mask.append(len(pos) > 0)

print(f"Items with >=0.8 positive under threshold {JACCARD_BINARY_THRESHOLD}: {sum(kept_mask)} / {len(eval_items)}")


Items with >=0.8 positive under threshold 0.8: 1074 / 1074


### 6. Precompute answers

In [8]:
def build_sorted_pools_from_emb_map(
    emb_map: Dict[str, np.ndarray],
    items: List[Tuple[str, List[str]]]
) -> List[List[str]]:
    sorted_pools: List[List[str]] = []
    for (anchor_q, pool) in items:
        a_emb = emb_map[anchor_q]
        pool_embs = np.vstack([emb_map[q] for q in pool])
        sims = pool_embs @ a_emb
        order = np.argsort(-sims)
        sorted_pools.append([pool[i] for i in order])
    return sorted_pools

def avg_jaccard_at_k(sorted_pools: List[List[str]], k: int) -> float:
    total = 0.0
    count = 0
    for sorted_pool, rel_map in zip(sorted_pools, jaccard_scores_per_item):
        topk = sorted_pool[:k]
        if not topk:
            continue
        total += float(np.mean([rel_map[q] for q in topk]))
        count += 1
    return total / max(count, 1)

def dcg(rels: List[float]) -> float:
    return float(sum((r / np.log2(i + 2)) for i, r in enumerate(rels)))

def avg_ndcg_at_k(sorted_pools: List[List[str]], k: int) -> float:
    total = 0.0
    count = 0
    for sorted_pool, rel_map in zip(sorted_pools, jaccard_scores_per_item):
        topk = sorted_pool[:k]
        if not topk:
            continue

        rels = [rel_map[q] for q in topk]
        dcg_val = dcg(rels)

        ideal_rels = sorted(rel_map.values(), reverse=True)[:k]
        idcg = dcg(ideal_rels)

        ndcg = (dcg_val / idcg) if idcg > 0 else 0.0
        total += ndcg
        count += 1

    return total / max(count, 1)

def avg_prf_at_k(sorted_pools: List[List[str]], k: int) -> Tuple[float, float, float]:
    p_sum = r_sum = f1_sum = 0.0
    total = 0

    for sorted_pool, pos_set, keep in zip(sorted_pools, pos_sets, kept_mask):
        if not keep:
            continue

        topk_set = set(sorted_pool[:k])
        inter = len(topk_set & pos_set)

        p = inter / k if k > 0 else 0.0
        r = inter / len(pos_set) if len(pos_set) > 0 else 0.0
        f1 = (2*p*r/(p+r)) if (p+r) > 0 else 0.0

        p_sum += p
        r_sum += r
        f1_sum += f1
        total += 1

    if total == 0:
        return 0.0, 0.0, 0.0
    return p_sum/total, r_sum/total, f1_sum/total

def avg_spearman(sorted_pools: List[List[str]]) -> float:
    """Mean Spearman rank correlation: model ranking vs Jaccard ranking of the pool."""
    rhos = []
    for sorted_pool, rel_map in zip(sorted_pools, jaccard_scores_per_item):
        n = len(sorted_pool)
        if n < 2:
            continue
        model_ranks = np.arange(1, n + 1, dtype=float)
        sorted_by_jaccard = sorted(sorted_pool, key=lambda q: rel_map[q], reverse=True)
        q_to_jrank = {q: r for r, q in enumerate(sorted_by_jaccard, start=1)}
        jaccard_ranks = np.array([q_to_jrank[q] for q in sorted_pool])
        rho, _ = spearmanr(model_ranks, jaccard_ranks)
        if not np.isnan(rho):
            rhos.append(rho)
    return float(np.mean(rhos)) if rhos else 0.0

def report(name: str, sorted_pools: List[List[str]]):
    print(f"\n{name}")
    sp = avg_spearman(sorted_pools)
    print(f"Spearman (rank corr.): {sp:.4f}")
    print("Graded relevance (Jaccard):")
    for k in TOPKS:
        aj = avg_jaccard_at_k(sorted_pools, k)
        nd = avg_ndcg_at_k(sorted_pools, k)
        print(f"@{k:<2} | avgJacc: {aj:.4f} | nDCG: {nd:.4f}")
    print(f"\nBinary relevance (Jaccard >= {JACCARD_BINARY_THRESHOLD}):")
    for k in TOPKS:
        p, r, f1 = avg_prf_at_k(sorted_pools, k)
        print(f"@{k:<2} | P: {p:.4f} | R: {r:.4f} | F1: {f1:.4f}")


In [9]:
qwen = SentenceTransformer(QWEN_MODEL_NAME)
qwen_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_embs = qwen.encode(batch_qs, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        qwen_emb_map[q] = e
qwen_sorted_pools = build_sorted_pools_from_emb_map(qwen_emb_map, eval_items)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 395.14it/s, Materializing param=norm.weight]                              


In [10]:
minilm = SentenceTransformer(MINILM_MODEL_NAME)
minilm_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_embs = minilm.encode(batch_qs, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        minilm_emb_map[q] = e
minilm_sorted_pools = build_sorted_pools_from_emb_map(minilm_emb_map, eval_items)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 379.35it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 8. Ranking helper and metrics

In [11]:
e5 = SentenceTransformer(E5_MODEL_NAME)

def e5_query_text(s: str) -> str:
    return f"query: {s}"

e5_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_pref = [e5_query_text(q) for q in batch_qs]
    batch_embs = e5.encode(batch_pref, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        e5_emb_map[q] = e
e5_sorted_pools = build_sorted_pools_from_emb_map(e5_emb_map, eval_items)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 386.36it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
mpnet = SentenceTransformer(MPNET_MODEL_NAME)
mpnet_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_embs = mpnet.encode(batch_qs, convert_to_numpy=True, normalize_embeddings=True)
    for q, e in zip(batch_qs, batch_embs):
        mpnet_emb_map[q] = e
mpnet_sorted_pools = build_sorted_pools_from_emb_map(mpnet_emb_map, eval_items)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 386.83it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
X = tfidf.fit_transform(unique_queries)
X = sk_normalize(X, norm="l2", axis=1)

q_to_row = {q: i for i, q in enumerate(unique_queries)}

tfidf_sorted_pools: List[List[str]] = []
for (anchor_q, pool) in eval_items:
    a_row = q_to_row[anchor_q]
    a_vec = X[a_row]
    pool_rows = [q_to_row[q] for q in pool]
    pool_vecs = X[pool_rows]
    sims = (pool_vecs @ a_vec.T).toarray().ravel()
    order = np.argsort(-sims)
    tfidf_sorted_pools.append([pool[i] for i in order])


In [14]:
hybrid = SPARQLEmbedder(CHECKPOINT_PATH)
hybrid_emb_map: Dict[str, np.ndarray] = {}
for i in range(0, len(unique_queries), BATCH):
    batch_qs = unique_queries[i:i+BATCH]
    batch_embs = hybrid.encode(batch_qs)
    batch_embs = batch_embs / (np.linalg.norm(batch_embs, axis=1, keepdims=True) + 1e-12)
    for q, e in zip(batch_qs, batch_embs):
        hybrid_emb_map[q] = e
hybrid_sorted_pools = build_sorted_pools_from_emb_map(hybrid_emb_map, eval_items)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 526.98it/s, Materializing param=pooler.dense.weight]                               


Loading model from ../checkpoints/wikidata_model_semantic-20.pt...


In [15]:
report("Baseline: Qwen", qwen_sorted_pools)
report("Baseline: MiniLM", minilm_sorted_pools)
report("Baseline: E5", e5_sorted_pools)
report("Baseline: MPNet", mpnet_sorted_pools)
report("Baseline: TF-IDF", tfidf_sorted_pools)
report("HybridSPARQLEncoder", hybrid_sorted_pools)
print("\nCached unique queries:", len(answer_cache))



Baseline: Qwen
Spearman (rank corr.): 0.9665
Graded relevance (Jaccard):
@3  | avgJacc: 0.5537 | nDCG: 0.7966
@5  | avgJacc: 0.4597 | nDCG: 0.8169
@10 | avgJacc: 0.3570 | nDCG: 0.8563

Binary relevance (Jaccard >= 0.8):
@3  | P: 0.5667 | R: 0.6119 | F1: 0.4728
@5  | P: 0.4698 | R: 0.7181 | F1: 0.4475
@10 | P: 0.3637 | R: 0.8760 | F1: 0.4081

Baseline: MiniLM
Spearman (rank corr.): 0.9653
Graded relevance (Jaccard):
@3  | avgJacc: 0.5409 | nDCG: 0.7554
@5  | avgJacc: 0.4687 | nDCG: 0.7938
@10 | avgJacc: 0.3573 | nDCG: 0.8280

Binary relevance (Jaccard >= 0.8):
@3  | P: 0.5537 | R: 0.6004 | F1: 0.4618
@5  | P: 0.4786 | R: 0.7350 | F1: 0.4588
@10 | P: 0.3638 | R: 0.8807 | F1: 0.4116

Baseline: E5
Spearman (rank corr.): 0.9638
Graded relevance (Jaccard):
@3  | avgJacc: 0.5555 | nDCG: 0.7897
@5  | avgJacc: 0.4643 | nDCG: 0.8133
@10 | avgJacc: 0.3534 | nDCG: 0.8476

Binary relevance (Jaccard >= 0.8):
@3  | P: 0.5683 | R: 0.6053 | F1: 0.4700
@5  | P: 0.4745 | R: 0.7217 | F1: 0.4516
@10 | P: 